# 🎬 London Beer Flood 1814 - Fast Cloud Render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunmalpani/london-beer-flood-1814/blob/main/render_colab.ipynb)

### ⚡ Quick Start:
1. **Stop previous run** (if still running) by clicking the square Stop button on the running cell.
2. Run all cells: click **Runtime** &rarr; **Run all** (or `Ctrl + F9`).
3. Render will finish in **~2-4 minutes**!
4. The final video `final_beer_flood_1814.mp4` will download automatically.

In [ ]:
# Step 1: Check System
!nvidia-smi

In [ ]:
# Step 2: Pull latest code & install dependencies
import os
if os.path.exists('london-beer-flood-1814'):
    %cd london-beer-flood-1814
    !git pull origin main
else:
    !git clone https://github.com/arjunmalpani/london-beer-flood-1814.git
    %cd london-beer-flood-1814

!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs ffmpeg xvfb libegl1-mesa-dev libgl1-mesa-dev
!npm install --legacy-peer-deps

In [ ]:
# Step 3: Test still frame 100
!mkdir -p out
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion still src/index.ts BeerFlood out/test_gpu.jpg \
    --frame=100 \
    --timeout=60000 \
    --overwrite
!ls -lh out/test_gpu.jpg

In [ ]:
# Step 4: Render all 2,092 frames (Ultra-Fast: ~2-3 minutes)
!xvfb-run -a -s "-screen 0 1920x1080x24" npx remotion render src/index.ts BeerFlood out/raw_video.mp4 \
    --concurrency=4 \
    --timeout=120000 \
    --codec=h264 \
    --crf=18 \
    --overwrite

In [ ]:
# Step 5: Mux master audio mix (-15 LUFS stereo) & verify
!ffmpeg -y -i out/raw_video.mp4 -i public/audio/mix.wav \
    -map 0:v:0 -map 1:a:0 \
    -c:v copy \
    -c:a aac -b:a 320k \
    -shortest \
    out/final_beer_flood_1814.mp4

!ls -lh out/final_beer_flood_1814.mp4
print("\n Render and Master Audio Muxing Completed Successfully!")

In [ ]:
# Step 6: Download the finished video
from google.colab import files
files.download('out/final_beer_flood_1814.mp4')